# Tome 2, Chapter 11: Representations and Characters (Python companion)

Exercises first, solutions at the end. We check the orthogonality relations numerically, build the icosahedral representation of $A_5$ and watch the golden ratio appear, compute character tables *automatically* from class sums (the Burnside–Dixon method), implement the fast Fourier transform as harmonic analysis on $\Z/2^k\Z$, and use Dirichlet characters to count primes in progressions.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import cmath
import time
from collections import Counter
import numpy as np
import sympy
from sympy.combinatorics import Permutation
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Exercises

**P11.1 (★) Orthogonality, numerically.** Enumerate $\mathfrak S_4$, compute its conjugacy classes, enter the five rows of the character table of Figure 11.1, and check row and column orthogonality (Theorem 11.8, Corollary 11.11). Then decompose the permutation representation on the $6$ pairs (Exercise 11.1) and on the $24$ elements (the regular representation).

In [ ]:
# Your code here


**P11.2 (★★) The icosahedron and the golden ratio.** Generate the rotation group of the icosahedron from a rotation of order $5$ about a vertex and a rotation of order $3$ about a face centre (as $3\times3$ matrices), check that it has $60$ elements, sort them by rotation angle, and compare the traces with the row $\psi_3$ of the table of $A_5$.

In [ ]:
# Your code here


**P11.3 (★★★) Character tables from class sums.** For a group given by permutations, the class sums $e_C=\sum_{x\in C}x$ satisfy $e_Ce_D=\sum_Ea_{CDE}e_E$, and an irreducible character gives a common eigenvector $(\omega_C)_C$ of the matrices $(a_{CDE})_{D,E}$ (Exercise 11.8). Compute the matrices for $\mathfrak S_4$ and $A_5$, find their common eigenvectors (diagonalise a random combination), and recover the character tables, with $\chi(g_C)=d\,\omega_C/|C|$ and $d$ fixed by $\langle\chi,\chi\rangle=1$. This is the idea behind the algorithms computer algebra systems use.

In [ ]:
# Your code here


**P11.4 (★) The fast Fourier transform.** Implement the discrete Fourier transform $\hat f(j)=\sum_kf(k)e^{-2i\pi jk/n}$ naively, then recursively for $n=2^m$ (split $k$ into even and odd, i.e.\ restrict to the subgroup $2\Z/n\Z$ and its coset). Compare with `numpy.fft.fft`, time them for $n=2^{10}$, and check the convolution theorem (Exercise 11.5).

In [ ]:
# Your code here


**P11.5 (★★) Primes in progressions.** Count the primes $p\le 10^6$ in each class modulo $5$ and modulo $8$, and compute the character sums $S(\chi)=\sum_{p\le x}\chi(p)$. Use the filter of Proposition 11.15 to recover the counts from the $S(\chi)$. Which classes lead, and what does $S(\chi)$ for the non-principal characters look like compared with $S(\chi_0)$?

In [ ]:
# Your code here


---
## Solutions

### P11.1

In [ ]:
G = [Permutation(list(p)) for p in itertools.permutations(range(4))]
def classes_of(G):
    seen, classes = set(), []
    for g in G:
        if g in seen: continue
        C = {h * g * h**-1 for h in G}
        seen |= C; classes.append(sorted(C, key=lambda x: x.array_form))
    return classes
classes = classes_of(G)
def ctype(g):
    return tuple(sorted((len(c) for c in g.full_cyclic_form), reverse=True))
order = [(1, 1, 1, 1), (2, 1, 1), (2, 2), (3, 1), (4,)]
classes.sort(key=lambda C: order.index(ctype(C[0])))
sizes = np.array([len(C) for C in classes])
T = np.array([[1, 1, 1, 1, 1], [1, -1, 1, 1, -1], [2, 0, 2, -1, 0], [3, 1, -1, 0, -1], [3, -1, -1, 0, 1]], dtype=float)
print('class sizes', sizes)
print('row orthogonality (should be I):'); print(T @ np.diag(sizes) @ T.T / 24)
print('column orthogonality (should be diag of centraliser orders):'); print(T.T @ T, ' centralisers', 24 // sizes)

def perm_character(action, X):
    return np.array([sum(1 for x in X if action(C[0], x) == x) for C in classes], dtype=float)
pairs = [frozenset(p) for p in itertools.combinations(range(4), 2)]
chi_pairs = perm_character(lambda g, x: frozenset(g(i) for i in x), pairs)
chi_reg = perm_character(lambda g, x: g * x, G)
for name, chi in [('pairs', chi_pairs), ('regular', chi_reg)]:
    print(name, chi, '-> multiplicities', T @ (sizes * chi) / 24)

The rows are orthonormal for the weighted inner product, and $T^{\mathsf T}T$ is diagonal with entries $24,4,8,3,4$, the centraliser orders. The pairs give $\mathbf1+\chi_2+\chi_3$, as in Exercise 11.1; the regular representation contains each irreducible as many times as its dimension: $1,1,2,3,3$.

### P11.2

In [ ]:
phi = (1 + 5**0.5) / 2
def rotation(axis, angle):
    a = np.array(axis, dtype=float); a /= np.linalg.norm(a)
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    return np.eye(3) + math.sin(angle) * K + (1 - math.cos(angle)) * K @ K    # Rodrigues' formula
# vertices of the icosahedron: cyclic permutations of (0, +-1, +-phi)
verts = [np.array(v, dtype=float) for s1 in (1, -1) for s2 in (1, -1) for v in ((0, s1, s2 * phi), (s1, s2 * phi, 0), (s2 * phi, 0, s1))]
v0 = verts[0]
nbrs = [v for v in verts if abs(np.linalg.norm(v - v0) - 2) < 1e-9]           # edges have length 2
face = next(c for c in itertools.combinations(nbrs, 2) if abs(np.linalg.norm(c[0] - c[1]) - 2) < 1e-9)
r5 = rotation(v0, 2 * math.pi / 5)
r3 = rotation(v0 + face[0] + face[1], 2 * math.pi / 3)
group = [np.eye(3)]
frontier = [np.eye(3)]
key = lambda M: tuple(np.round(M, 6).ravel())
seen = {key(np.eye(3))}
while frontier:
    new = []
    for M in frontier:
        for g in (r5, r3):
            P = g @ M
            if key(P) not in seen:
                seen.add(key(P)); group.append(P); new.append(P)
    frontier = new
print('order of the group:', len(group))
print('preserves the vertex set:', all(any(np.allclose(M @ v, w) for w in verts) for M in group for v in verts))
traces = Counter(round(float(np.trace(M)), 6) for M in group)
print('traces and how often they occur:', dict(traces))
print('phi =', round(phi, 6), ' 1 - phi =', round(1 - phi, 6))

$60$ rotations, with traces $3$ (once, the identity), $-1$ ($15$ half-turns about edge midpoints), $0$ ($20$ rotations by $\pm120^\circ$ about face centres), $\varphi=1+2\cos72^\circ$ ($12$ rotations by $\pm72^\circ$) and $1-\varphi=1+2\cos144^\circ$ ($12$ rotations by $\pm144^\circ$). These are the class sizes $1,15,20,12,12$ of $A_5$ and the values of $\psi_3$: the rotation group of the icosahedron is $A_5$ (proved in Chapter 12), and its natural representation is irreducible because $\langle\psi_3,\psi_3\rangle=\frac{9+15+0+12\varphi^2+12(1-\varphi)^2}{60}=1$.

### P11.3

In [ ]:
def character_table(G, seed=0):
    classes = classes_of(G)
    n, r = len(G), len(classes)
    index = {g: i for i, C in enumerate(classes) for g in C}
    # a[C][D][E] = number of (x, y) in C x D with x*y = z, for a fixed z in E
    a = np.zeros((r, r, r))
    for c, C in enumerate(classes):
        for d, D in enumerate(classes):
            cnt = Counter(index[x * y] for x in C for y in D)
            for e in range(r):
                a[c, d, e] = cnt[e] / len(classes[e])
    rng = np.random.default_rng(seed)
    coeffs = rng.normal(size=r)
    # omega_C omega_D = sum_E a[C,D,E] omega_E: (omega_D)_D is an eigenvector of every matrix a[C], eigenvalue omega_C
    M = sum(coeffs[c] * a[c] for c in range(r))
    w, V = np.linalg.eig(M)
    sizes = np.array([len(C) for C in classes])
    table = []
    for k in range(r):
        omega = V[:, k] / V[0, k]          # omega of the class {e} is 1
        x = omega / sizes                  # chi(g_C) / d
        d = math.sqrt(n / np.sum(sizes * np.abs(x)**2))
        table.append(d * x)
    table = np.array(sorted(table, key=lambda row: (round(row[0].real), -row.real.sum())))
    return classes, sizes, table

for name, H in [('S4', G), ('A5', [Permutation(list(p)) for p in itertools.permutations(range(5)) if Permutation(list(p)).is_even])]:
    cl, sz, tab = character_table(H)
    print(name, 'class sizes', sz, ' cycle types', [ctype(C[0]) for C in cl])
    print(np.real_if_close(np.round(tab, 4)))

The method needs nothing but the multiplication table: it finds $5$ characters for each group, with dimensions $1,1,2,3,3$ and $1,3,3,4,5$ in the first column, and for $A_5$ the golden ratio $1.618$ and $-0.618$ appear by themselves (rows and columns come out in the order the computation produces them). Dixon (1967) made this exact by working modulo a suitable prime, and it is the core of the character table algorithms in GAP and Magma.

### P11.4

In [ ]:
def dft_naive(f):
    n = len(f)
    return [sum(f[k] * cmath.exp(-2j * math.pi * j * k / n) for k in range(n)) for j in range(n)]

def fft(f):
    n = len(f)
    if n == 1:
        return list(f)
    even, odd = fft(f[0::2]), fft(f[1::2])        # the subgroup 2Z/nZ and its coset
    out = [0] * n
    for j in range(n // 2):
        t = cmath.exp(-2j * math.pi * j / n) * odd[j]
        out[j], out[j + n // 2] = even[j] + t, even[j] - t
    return out

rng = np.random.default_rng(1)
f = list(rng.normal(size=2**10))
for name, F in [('naive', dft_naive), ('recursive FFT', fft), ('numpy', lambda x: list(np.fft.fft(x)))]:
    t0 = time.perf_counter(); out = F(f); t1 = time.perf_counter()
    print(f'{name:14s} {t1 - t0:8.4f} s   max error vs numpy {max(abs(a - b) for a, b in zip(out, np.fft.fft(f))):.2e}')

n = 64
g, h = rng.normal(size=n), rng.normal(size=n)
conv = np.array([sum(g[j] * h[(k - j) % n] for j in range(n)) for k in range(n)])
print('convolution theorem:', np.allclose(np.fft.fft(conv), np.fft.fft(g) * np.fft.fft(h)))
c = rng.normal(size=n)
C = np.array([[c[(k - j) % n] for j in range(n)] for k in range(n)])
lam = np.fft.fft(c)
chis = [np.exp(2j * np.pi * m * np.arange(n) / n) for m in range(n)]
print('each character is an eigenvector of C, eigenvalue = DFT of c:', all(np.allclose(C @ chis[m], lam[m] * chis[m]) for m in range(n)))

Here the transform is not divided by $n$ (the convention of numpy; the chapter's $\hat f$ is this divided by $n$), so the convolution theorem reads $\widehat{g*h}=\hat g\,\hat h$. The recursion splits $\Z/n\Z$ along the subgroup of index $2$: a character of $\Z/n\Z$ restricted to the even numbers is a character of $\Z/(n/2)\Z$, and $\chi(2k+1)=\chi(1)\chi(2k)$. Each level costs $O(n)$, and there are $\log_2n$ levels. Even in pure Python the recursive version beats the naive $O(n^2)$ sum by a factor of several hundred at $n=1024$, and numpy's compiled FFT is faster still.

### P11.5

In [ ]:
N = 10**6
primes = list(sympy.primerange(2, N + 1))
for q, gen_vals in [(5, None), (8, None)]:
    units = [a for a in range(1, q) if math.gcd(a, q) == 1]
    counts = Counter(p % q for p in primes if q % p != 0)
    print(f'mod {q}: counts', {a: counts[a] for a in units})
# characters modulo 5: chi_j(2^k) = i^(jk)
dlog5 = {1: 0, 2: 1, 4: 2, 3: 3}
chars5 = [{a: 1j**(j * dlog5[a]) for a in dlog5} for j in range(4)]
S = [sum(chi[p % 5] for p in primes if p != 5) for chi in chars5]
print('character sums mod 5:', [complex(round(s.real), round(s.imag)) for s in S])
recovered = {a: round((sum(chi[a].conjugate() * s for chi, s in zip(chars5, S)) / 4).real) for a in dlog5}
print('recovered counts     :', recovered)

xs = list(range(1000, N + 1, 1000))
idx = np.searchsorted(primes, xs, side='right')
leg = np.cumsum([0] + [1 if p % 5 in (1, 4) else (-1 if p % 5 in (2, 3) else 0) for p in primes])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(xs, leg[idx], lw=0.8, label='$S(\\chi_2)$, the Legendre symbol mod 5')
ax.axhline(0, lw=0.5, color='grey')
ax.set_xlabel('x'); ax.legend(); plt.show()

The four classes modulo $5$ receive almost exactly a quarter of the primes each (about $19\,620$), and similarly modulo $8$. Equivalently, the character sums $S(\chi)$ of the non-principal characters are tiny (a few dozen) compared with $S(\chi_0)=\pi(10^6)-1=78\,497$: that $S(\chi)=o(S(\chi_0))$ for every $\chi\ne\chi_0$ is the prime number theorem for arithmetic progressions, and the fact that $\sum_p\chi(p)/p$ converges, which already gives infinitely many primes in each class, is Dirichlet's theorem (Tome 5). The filter recovers the counts exactly from the four sums.

The plot shows a slight preference: $S(\chi_2)$, which counts primes that are squares modulo $5$ minus non-squares, is mostly negative. This is *Chebyshev's bias*: non-residues are ahead most of the time, a phenomenon explained by the zeros of $L$-functions (Rubinstein and Sarnak, 1994).